# Life As An AI Engineer — Setup & Core Concepts

## 🎯 By the end of this session you will have

- Understood the course structure, the technical challenges, and what certification requires
- Met the cohort staff, your peer supporters, and your journey group
- **Set up and validated** your AI engineering development environment
- Applied AI-assisted coding practice using Claude Code

The third one is the deliverable. The rest you get by showing up; a working machine you
have to actually build, and this session exists so that you do it **now** rather than in
week 4 with an assignment due.

---

## 🏗️ Build | 🚢 Ship | 📤 Share

**This notebook is how you earn that deliverable.** The outcomes above are the session;
below is what the notebook itself covers, what counts as done, and what to post when it is.

### 🏗️ Build
- This repository understood as a **git** — upstream, origin, local — with the commands run live
- Your first LLM API call, and the roles that every LLM message is built from
- Five prompt-engineering patterns, each shown weak→strong so you can see *why* it matters
- The frontend/backend split that every app in this course will use

### 🚢 Ship
Get every cell in this notebook running on your machine — that is the validated environment,
proven rather than assumed: not a clever app, but a machine you can build on for the next ten weeks.

### 📤 Share
Post the output of Part 0 (your environment check) in the cohort channel. If something
failed, post that instead — a broken setup found in week 1 is cheap, and in week 4 is not.

---

> **This notebook assumes nothing is installed yet.** Part 0 runs without an API key,
> so you can check your environment before you spend a cent. Everything from Part 2
> onward needs a key.

## How to use this notebook

| | |
|---|---|
| **Where to run it** | From the `01_Life_As_An_AI_Engineer/` folder, with `.venv` selected as your kernel |
| **Setup** | `uv sync`, then `cp .env.example .env` and add your OpenAI key |
| **Order** | Top to bottom. Part 0 needs no key; Parts 2+ do |
| **If a cell fails** | Read the error, then check the "If it's blocked" section of this folder's `README.md` |

Full install instructions, if you have not done the one-click setup yet:
[Initial Setup](../00_Docs/Prerequisites/Initial_Setup/README.md).


## Contents

**[Part 0 — Does your environment actually work?](#part-0)**  
checks: 0.1 Python · 0.2 packages · 0.3 CLI tools · 0.4 API key · 0.5 installer layout · 0.6 installer's own check

**[Part 1 — Version control](#part-1)**  
[Why version control](#why-version-control) · [How — this repository is a git](#how-git)

**[Part 2 — The LLM API call](#part-2)**  
[Why call an API at all?](#why-api) · [The three roles](#three-roles) · [Setup — the client and two helpers](#api-setup)

**[Part 3 — Prompt engineering, five patterns](#part-3)**  
[1 Persona](#pattern-1) · [2 Few-shot](#pattern-2) · [3 Chain of thought](#pattern-3) · [4 Structured output](#pattern-4) · [5 Grounded context](#pattern-5)

**[Part 4 — Frontend and backend](#part-4)**  
[Why split them at all?](#why-split) · [How — no demo, by design](#how-no-demo)

**[🏗️ Activity — stack three patterns](#activity)** · **[✅ Done when](#done-when)**

---

<a id="part-0"></a>
# Part 0 — Does your environment actually work?

> ### Why this comes first
>
> Every hour lost in this course to "it works on my machine" is lost here, in week 1,
> to something small: a Python that is too old, a kernel pointing at the wrong
> interpreter, a key in the wrong file. The five cells below check each of those
> **separately**, so a failure tells you *which* thing is broken.
>
> This part needs no API key and costs nothing.

### 0.1 — Which Python is this notebook actually running?

This is the single most common setup failure in the course, and it is not obvious:
you can have a perfect `.venv` on disk and still have the notebook attached to your
*system* Python. The two paths below have to point into this folder's `.venv`.

If they don't: in VS Code, click the kernel picker (top right) and choose the
interpreter at `01_Life_As_An_AI_Engineer/.venv/bin/python`.


In [1]:
import sys, platform
from pathlib import Path

print(f"Python version   : {platform.python_version()}")
print(f"Executable       : {sys.executable}")
print(f"Prefix           : {sys.prefix}")
print()

# The course pins >=3.12,<3.15 (see pyproject.toml).
major, minor = sys.version_info[:2]
ok_version = (major == 3) and (12 <= minor < 15)
print(f"{'✅' if ok_version else '❌'} Version in range >=3.12,<3.15")

# Is this interpreter the one inside THIS folder's .venv?
in_venv = ".venv" in sys.prefix
here_venv = (Path.cwd() / ".venv").resolve()
is_local = Path(sys.prefix).resolve() == here_venv
print(f"{'✅' if in_venv else '❌'} Running inside a virtual environment")
print(f"{'✅' if is_local else '⚠️ '} That venv is this folder's .venv")
if in_venv and not is_local:
    print(f"     expected: {here_venv}")
    print(f"     actual  : {Path(sys.prefix).resolve()}")
    print("     -> switch your kernel, or you'll install packages into the wrong place")


Python version   : 3.12.13
Executable       : c:\Users\Omart\Code\work\AIE11\01_Life_As_An_AI_Engineer\.venv\Scripts\python.exe
Prefix           : c:\Users\Omart\Code\work\AIE11\01_Life_As_An_AI_Engineer\.venv

✅ Version in range >=3.12,<3.15
✅ Running inside a virtual environment
✅ That venv is this folder's .venv


### 0.2 — Are the packages importable?

`uv sync` installs them; this proves the *running kernel* can see them. A failure here
with a passing 0.1 almost always means you ran `uv sync` in a different folder.


In [2]:
import importlib

required = {
    "openai":   "the LLM client",
    "dotenv":   "reads your .env file (package name: python-dotenv)",
    "pydantic": "structured output schemas (Part 3)",
}

missing = []
for mod, why in required.items():
    try:
        m = importlib.import_module(mod)
        version = getattr(m, "__version__", "?")
        print(f"✅ {mod:<10} {version:<12} {why}")
    except ImportError:
        missing.append(mod)
        print(f"❌ {mod:<10} {'MISSING':<12} {why}")

if missing:
    print()
    print("Fix: run `uv sync` in 01_Life_As_An_AI_Engineer/, then restart the kernel.")


✅ openai     3.48.0       the LLM client
✅ dotenv     ?            reads your .env file (package name: python-dotenv)
✅ pydantic   2.14.3       structured output schemas (Part 3)


### 0.3 — Are the command-line tools there?

`git`, `uv`, `node` and `claude` are the tools you will use every week. We ask the
**shell**, not Python, because the shell is where you will actually type them.

A "not found" here almost never means the tool is missing. It means the tools are
installed but **not loaded into this terminal**. The course installer deliberately does
not touch every shell on your machine; you load it per terminal:

```bash
source ~/ai-eng/env.sh          # Mac/Linux
. $HOME\ai-eng\env.ps1          # Windows
```

That line goes in **every new terminal** you open for coursework. The one exception is
VS Code's `code` command, which is configured once and works everywhere.

> If you launched this notebook from a terminal that had not been loaded, the kernel
> inherited that environment — so load it, then **restart the kernel**.


In [3]:
import shutil, subprocess

def tool_version(name, args):
    """Return a one-line version string, or None if the tool isn't on PATH."""
    if shutil.which(name) is None:
        return None
    try:
        out = subprocess.run([name, *args], capture_output=True, text=True, timeout=15)
        return (out.stdout or out.stderr).strip().splitlines()[0]
    except Exception as exc:
        return f"(found, but failed to run: {exc})"

for name, args, needed in [
    ("git",    ["--version"], "required"),
    ("uv",     ["--version"], "required"),
    ("claude", ["--version"], "recommended"),
    ("node",   ["--version"], "needed later, for frontends"),
]:
    v = tool_version(name, args)
    mark = "✅" if v else ("❌" if needed == "required" else "⚠️ ")
    print(f"{mark} {name:<8} {v or 'not on PATH'}   ({needed})")


✅ git      git version 2.55.0.windows.3   (required)
✅ uv       uv 0.12.21 (7af826859 2026-09-29 x86_64-pc-windows-msvc)   (required)
✅ claude   2.1.316 (Claude Code)   (recommended)
✅ node     v24.18.0   (needed later, for frontends)


### 0.4 — Is your API key loaded?

Two separate things have to be true: the `.env` file exists, **and** this process read it.
We check both, and we never print the key itself — only its shape.

> ⚠️ **Never print or paste a full API key**, into a notebook output, a screenshot, or a
> help channel. Notebook outputs get committed to git. The check below shows only the
> prefix and length, which is enough to debug and useless to a thief.


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

# This folder's .env takes precedence; the repo root is a fallback.
local_env = Path(".env")
root_env  = Path("..") / ".env"
load_dotenv(local_env)
load_dotenv(root_env)

print(f"{'✅' if local_env.exists() else '⚠️ '} ./.env                 {'found' if local_env.exists() else 'not found'}")
print(f"{'✅' if root_env.exists()  else '  '} ../.env (repo root)   {'found' if root_env.exists() else 'not found'}")
print()

key = os.environ.get("OPENAI_API_KEY", "")
if not key:
    print("❌ OPENAI_API_KEY is not set.")
    print("   Fix: cp .env.example .env    then edit .env and paste your key")
    print("   Get a key: https://platform.openai.com/api-keys")
elif key == "sk-your-key-here":
    print("❌ OPENAI_API_KEY is still the placeholder from .env.example.")
    print("   Fix: edit .env and paste your real key")
else:
    print(f"✅ OPENAI_API_KEY loaded  (starts {key[:7]}..., {len(key)} chars)")


✅ ./.env                 found
   ../.env (repo root)   not found

✅ OPENAI_API_KEY loaded  (starts sk-proj..., 164 chars)


### 0.5 — What the course installer put on your machine

Everything above tested *this notebook*. This cell looks at what the
[course environment installer](https://github.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers)
actually created, so you can see the layout rather than trust it.

Everything the installer manages lives in **one directory**, `~/ai-eng`, plus Claude Code
in `~/.local/bin`. That is deliberate: nothing is scattered across your machine, and
removing that one directory undoes almost all of it.

In [5]:
from pathlib import Path

AI_ENG = Path(os.environ.get("AI_ENG_HOME", Path.home() / "ai-eng"))
print(f"Installer home: {AI_ENG}")
print(f"{'✅ found' if AI_ENG.is_dir() else '❌ not found'}\n")

if not AI_ENG.is_dir():
    print("The course installer has not run on this machine.")
    print("Run it — see the commands in the next cell, or this folder's README.md.")
else:
    # What the installer writes, and what each piece is for.
    expected = {
        "env.sh":           "load the tools into a terminal  (Mac/Linux)",
        "env.ps1":          "load the tools into a terminal  (Windows)",
        "setup.sh":         "the installer itself            (Mac/Linux)",
        "setup.ps1":        "the installer itself            (Windows)",
        "README.txt":       "installer notes, rewritten every run",
        "setup.log":        "full log of the last run",
        "setup-report.txt": "written only when something FAILed or WARNed",
        "bin":              "uv lives here",
        "python":           "Python installs (UV_PYTHON_INSTALL_DIR)",
        "nvm":              "nvm, for Node                   (Mac/Linux)",
        "node":             "portable Node 24                (Windows)",
        "path-backup.txt":  "your PATH before it was edited  (Windows)",
    }
    for name, why in expected.items():
        p = AI_ENG / name
        print(f"  {'✅' if p.exists() else '  '} {name:<18} {why}")

    print()
    claude_bin = Path.home() / ".local" / "bin"
    print(f"  {'✅' if claude_bin.is_dir() else '  '} {'~/.local/bin':<18} Claude Code lives here (not in ai-eng)")

    # If the last run flagged anything, it left a report. Surface it.
    report = AI_ENG / "setup-report.txt"
    if report.exists():
        print(f"\n⚠️  A setup report exists, so the last run had a FAIL or WARN:")
        print("─" * 60)
        print(report.read_text(errors="replace")[:1500])
        print("─" * 60)
        print("The installer also copies this to your clipboard — paste it in the help channel.")
    else:
        print("\n✅ No setup-report.txt, which means the last run had no FAIL or WARN.")


Installer home: C:\Users\Omart\ai-eng
✅ found

     env.sh             load the tools into a terminal  (Mac/Linux)
     env.ps1            load the tools into a terminal  (Windows)
     setup.sh           the installer itself            (Mac/Linux)
  ✅ setup.ps1          the installer itself            (Windows)
     README.txt         installer notes, rewritten every run
     setup.log          full log of the last run
  ✅ setup-report.txt   written only when something FAILed or WARNed
     bin                uv lives here
     python             Python installs (UV_PYTHON_INSTALL_DIR)
     nvm                nvm, for Node                   (Mac/Linux)
     node               portable Node 24                (Windows)
     path-backup.txt    your PATH before it was edited  (Windows)

  ✅ ~/.local/bin       Claude Code lives here (not in ai-eng)

⚠️  A setup report exists, so the last run had a FAIL or WARN:
────────────────────────────────────────────────────────────
ï»¿AI Engineer set

### 0.6 — The installer's own check

Your machine is the installer's business, not this notebook's. When the two disagree,
the installer wins. Re-run it in **check mode**, which changes nothing:

```bash
bash ~/ai-eng/setup.sh --check                                          # Mac/Linux
powershell -ExecutionPolicy Bypass -File $HOME\ai-eng\setup.ps1 -Check   # Windows
```

You want **All good**. It exits non-zero if anything `[FAIL]`ed, so it also works in a script.

**Reading its output:**

| Marker | Means | Do |
|---|---|---|
| `[ OK ]` | already installed and working | nothing |
| `[DONE]` | installed or fixed on this run | nothing |
| `[ -- ]` | optional, skipped | nothing — this is fine |
| `[WARN]` | works, but something is worth a look | read the `->` hint under it |
| `[FAIL]` | needs fixing | read the `->` hint under it |

**Not installed yet?** The one-click install, from the course's
[installer repo](https://github.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers):

```bash
# Mac/Linux
mkdir -p ~/ai-eng && curl -fsSL https://raw.githubusercontent.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers/main/setup.sh -o ~/ai-eng/setup.sh && bash ~/ai-eng/setup.sh
```

```powershell
# Windows
mkdir -Force $HOME\ai-eng | Out-Null; irm https://raw.githubusercontent.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers/main/setup.ps1 -OutFile $HOME\ai-eng\setup.ps1; powershell -ExecutionPolicy Bypass -File $HOME\ai-eng\setup.ps1
```

It is **safe to re-run at any time** — anything already done is reported `[ OK ]` and
skipped. Useful flags:

| Mac/Linux | Windows | What it does |
|---|---|---|
| `--check` | `-Check` | report only, change nothing |
| `--tools-only` | `-ToolsOnly` | skip the Git name/email and GitHub SSH steps |
| `--yes` / `-y` | `-Yes` | ask nothing, take every default |
| `--no-copy` | `-NoCopy` | don't copy the report to your clipboard |
| `--help` / `-h` | — | print usage and exit |

Full setup walkthrough, including the VS Code install that comes first:
[Initial Setup](../00_Docs/Prerequisites/Initial_Setup/README.md) ·
[step-by-step DIY guide](https://github.com/AI-Maker-Space/Interactive-Dev-Environment-for-AI-Engineers/blob/main/details/README.md)

---

<a id="part-1"></a>
# Part 1 — Version control

<a id="why-version-control"></a>
## Why version control

Not "because you might lose your work." Backups solve that. Version control solves a
different problem: **more than one version of the truth has to exist at the same time.**

Picture one environment. You have an app that works, and you want to change it. The
moment you start editing, the working version is gone — you are editing the only copy
users have. Every change is a bet you cannot take back.

So you split the timeline:

| Stage | What it is | Who sees it | What is allowed to be true here |
|---|---|---|---|
| **dev** | your machine | you | broken, half-finished, experimental |
| **production** | what users use | everyone | only what works |

That is the two-stage model, and it is already enough to stop you breaking production
by typing. But it has a gap: the jump from "works on my laptop" to "live for everyone"
is one move, untested anywhere real.

So teams add a third:

| Stage | What it is | Who sees it | Why it exists |
|---|---|---|---|
| **dev** | your machine | you | write code |
| **staging** | a real deployment, fake users | the team | *this is where you find out your change was wrong* |
| **production** | the real thing | everyone | only what survived staging |

**Staging is the whole argument for version control.** If code can live in three states
at once, you need something that can hold three states at once and move changes between
them deliberately. That is git. The branches are not bureaucracy; they are the three
stages made concrete:

| Branch | Maps to | In this course |
|---|---|---|
| `main` | production. "The current official story" | where your submitted work lives |
| `develop` | staging. "The draft of the next chapter" | not used — we keep it simple |
| `feature/*` | dev. "Paragraphs before they go in the book" | use freely for experiments |

> **For this course we work directly on `main`.** That is a deliberate simplification,
> and it is the right call while you are learning. Know that the real version exists,
> and that the reason is staging. More: [Version Control](../00_Docs/Prerequisites/Initial_Setup/3_Version_Control/README.md).

<a id="how-git"></a>
## How version control — this repository is a git

Abstract diagrams of git do not stick. The repository you have open **is** a git, so
let's read it. Every cell below runs a real command against your actual clone.

### The three places your code lives

| Name | Where | What you do with it |
|---|---|---|
| **upstream** | AI Makerspace's repo | `pull` from it. Never push |
| **origin** | *your* fork on GitHub | `push` to it. This is what you submit |
| **local** | the folder on your laptop | edit files here |

Flow: `upstream` → `origin` → `local` → `origin`.

In [6]:
import subprocess
from pathlib import Path

REPO_ROOT = Path("..").resolve()

def git(*args, cwd=REPO_ROOT):
    """Run a git command and return its output as text (stdout+stderr)."""
    out = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    return (out.stdout + out.stderr).strip()

print(f"Repo root: {REPO_ROOT}")
print()
print("── Your remotes (git remote -v) ──")
print(git("remote", "-v") or "(no remotes configured)")


Repo root: C:\Users\Omart\Code\work\AIE11

── Your remotes (git remote -v) ──
origin	https://github.com/omar-paac/AIE11.git (fetch)
origin	https://github.com/omar-paac/AIE11.git (push)
upstream	https://github.com/AI-Maker-Space/AIE11.git (fetch)
upstream	DISABLE (push)


**Reading that output.** You should see `origin` twice (fetch and push). If you also see
`upstream`, your fork is wired up for course updates.

**No `upstream`?** That is the one-time setup you only do once per course:

```bash
git remote add upstream https://github.com/AI-Maker-Space/AIE11.git
git remote -v    # confirm it took
```

Full walkthrough: [Set Up Initial Repos](../00_Docs/Prerequisites/Initial_Setup/2_Set_Up_Initial_Repos/README.md).


In [7]:
print("── Where you are (git status -sb) ──")
print(git("status", "-sb"))
print()
print("── Recent history (git log --oneline -8) ──")
print(git("log", "--oneline", "-8"))


── Where you are (git status -sb) ──
## main...origin/main [ahead 2]

── Recent history (git log --oneline -8) ──
95c1d3d Session 2 - Quicklinks
0cd4c08 Session 2 - Quicklinks
d6bee95 AIE11 - Week 1 - Session 2
26e2a58 AIE11 - Week 1 - Session 2
7c25972 AIE11 - Week 1 - Session 2
2915a18 AIE11 - Week 1 - Session 2
5d4aeda Merge remote-tracking branch 'refs/remotes/origin/main'
b88a2cd AIE11 - Week 1 - Session 2


**Reading that output.**

- The first line of `status -sb` is your branch, and `[ahead N]` / `[behind N]` tells you
  how far local has drifted from origin. `ahead 2` means two commits you have not pushed.
- `??` marks files git has never seen. `M` marks tracked files you changed.
- Each `git log` line is one commit: a snapshot you can return to. That is the unit of
  work version control actually deals in.

### The commit you will make every week

A commit is three commands, and the middle one is the one people skip:

```bash
git add .                       # stage: choose what goes in
git diff --staged               # READ IT. this is the step that saves you
git commit -m "message"         # snapshot it
git push origin main            # send it to your fork
```

> ⚠️ **`git diff --staged` before every commit.** This is how you catch a `.env` with a
> live API key before it becomes permanent. A key pushed to a public repo is scraped in
> minutes, and deleting the commit does not help — you have to rotate the key.

Let's prove your `.env` is actually protected:


In [8]:
# Is .env ignored? Ask git, don't assume.
ignored = git("check-ignore", "-v", "01_Life_As_An_AI_Engineer/.env")
if ignored:
    print(f"✅ .env is gitignored by rule:\n   {ignored}")
else:
    print("⚠️  git does NOT report .env as ignored.")
    print("   Check .gitignore before you ever commit, or your key goes public.")

print()
print("── Anything currently staged that shouldn't be? ──")
staged = git("diff", "--staged", "--name-only")
print(staged if staged else "(nothing staged — clean)")


✅ .env is gitignored by rule:
   .gitignore:4:.env*	01_Life_As_An_AI_Engineer/.env

── Anything currently staged that shouldn't be? ──
(nothing staged — clean)


### Your weekly loop

```bash
git checkout main
git pull upstream main      # get new course material
git push origin main        # keep your fork level with it
# ... do the assignment ...
git add . && git diff --staged && git commit -m "week N" && git push origin main
```

That is the whole routine for ten weeks. Detail:
[Your Weekly Git Workflow](../00_Docs/Prerequisites/Initial_Setup/4_Your_Weekly_Git_Workflow/README.md).


---

<a id="part-2"></a>
# Part 2 — The LLM API call

<a id="why-api"></a>
## Why call an API at all?

You have used ChatGPT. Why not just use that?

Because a chat window is a **product**, and you are here to build products. The
difference is not the model — it is the same model. The difference is what you can do
with the call:

| In the chat window | Through the API |
|---|---|
| You type, you read | *Your code* types, and your code reads |
| One conversation | Ten thousand conversations, in parallel |
| You are the one deciding what to send | You decide the rules **before** the user arrives |
| Output is text on a screen | Output is a typed object your program acts on |

That third row is the real one. As a user you write a message. As an engineer you write
the **system instructions** — the rules that apply to every user, forever — and the
user's message arrives inside the frame you built. That asymmetry is the job.

> The same principle is why `AGENTS.md` in this repo can steer Claude Code into teaching
> mode. System-level instructions outrank what comes after them. You have already been
> on the engineering side of this without noticing.

<a id="three-roles"></a>
## The three roles

As a *user* of an LLM app, the shape is: text in, text out.

As a *developer*, it is: **a list of role-tagged messages in, one message out.**

| Role | Also called | Who writes it | What it does |
|---|---|---|---|
| `system` | developer | **you, the engineer** | the rules. Prioritised above everything else |
| `user` | human | your app's user (or you, faking one) | the request |
| `assistant` | AI | the model (or you, faking one) | a reply — and you can **put words in its mouth** |

That last one looks strange and is the key to few-shot prompting in Part 3: you write
fake `assistant` turns to demonstrate the behaviour you want, and the model continues
the pattern.

More: [LLM APIs — The "System" Role](../00_Docs/Prerequisites/Initial_Setup/6_LLM_APIs/README.md).

<a id="api-setup"></a>
## Setup — the client and two helpers

We use the OpenAI SDK against the **Responses API** (`client.responses.*`), the modern
surface for gpt-5-era models. This is the same helper layer as
[`instructor_materials/Prompt_Engineering_Patterns.ipynb`](instructor_materials/Prompt_Engineering_Patterns.ipynb) in this folder,
so the patterns you learn here carry straight over.

Override the model or endpoint in `.env` without touching code:

```
LLM_MODEL=gpt-5.4-mini                   # any OpenAI-compatible model string
LLM_API_BASE=http://localhost:4000/v1    # a LiteLLM proxy, a local server, ...
```

In [24]:
import os, time, json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv(Path(".env"))          # this folder's .env (preferred)
load_dotenv(Path("..") / ".env")   # ...or the repo root's
assert os.environ.get("OPENAI_API_KEY"), \
    "No OPENAI_API_KEY. Run `cp .env.example .env` and add your key, then restart the kernel."

MODEL    = os.getenv("LLM_MODEL", "gpt-5.4-mini")
API_BASE = os.getenv("LLM_API_BASE") or None
client   = OpenAI(base_url=API_BASE)

# ── Thin helpers ────────────────────────────────────────────────────────────

def ask(input, instructions=None, effort=None, **kw):
    """
    One call against the Responses API.
    `input`        — a plain string, or a list of role-message dicts
    `instructions` — the system / developer instruction string
    `effort`       — reasoning effort: None | 'low' | 'medium' | 'high'
                     (gpt-5+ models only; None = no explicit reasoning step)
    """
    params = {"model": MODEL, "input": input}
    if instructions is not None:
        params["instructions"] = instructions
    if effort is not None:
        params["reasoning"] = {"effort": effort}
    return client.responses.create(**params, **kw)

def show(resp) -> None:
    """Pretty-print a Response object's text into the notebook."""
    display(Markdown(resp.output_text))

print(f"✅ Model: {MODEL}   API base: {API_BASE or 'OpenAI default'}")


✅ Model: gpt-5.4-mini   API base: OpenAI default


### Your first call

One string in, one answer out. This is the whole API in its simplest form.


In [25]:
resp = ask("In two sentences, what does an AI engineer do that a data scientist doesn't?")
show(resp)


An AI engineer focuses on turning models into reliable, scalable products, handling deployment, integration, performance, monitoring, and maintenance in real-world systems. A data scientist is more focused on analyzing data, finding patterns, and building models or insights, often for experimentation and decision-making rather than full productionization.

### What came back besides the text

The text is what you read; the **metadata** is what you engineer against. Token counts
are what you pay for, and latency is what your user waits through. Look at both now, in
week 1, so the numbers in week 4 are not a surprise.


In [26]:
t0 = time.perf_counter()
resp = ask("Name the three message roles in an LLM API call. Just the three words.")
elapsed = time.perf_counter() - t0

print("── text ──")
print(resp.output_text)
print()
print("── metadata ──")
print(f"model          : {resp.model}")
print(f"latency        : {elapsed:.2f}s")
print(f"input tokens   : {resp.usage.input_tokens}")
print(f"output tokens  : {resp.usage.output_tokens}")
print(f"total tokens   : {resp.usage.total_tokens}")


── text ──
system, user, assistant

── metadata ──
model          : gpt-5.4-mini-2026-03-17
latency        : 1.29s
input tokens   : 23
output tokens  : 9
total tokens   : 32


### The three roles, for real

Now the developer-side shape: a **list** of role-tagged messages. Note the `assistant`
turn we wrote ourselves — the model has no idea we faked it, and it continues the
clipped style we demonstrated.


In [27]:
conversation = [
    {"role": "user",      "content": "What is uv?"},
    {"role": "assistant", "content": "A fast Python package and version manager. Replaces pip, venv, and pyenv."},
    {"role": "user",      "content": "And what is git?"},
]

print("The assistant turn above is OURS — we wrote it to set a terse style.")
print("Watch the answer below match that style without being asked:")
print("=" * 60)
show(ask(conversation))


The assistant turn above is OURS — we wrote it to set a terse style.
Watch the answer below match that style without being asked:


Git is a **version control system** for tracking changes to files, especially code.

It lets you:
- save snapshots of your project over time
- compare changes
- branch off to work on new features
- merge changes back together
- collaborate with others safely

Common terms:
- **commit**: a saved snapshot
- **branch**: an independent line of work
- **merge**: combine changes from branches
- **repository**: the project tracked by Git

In short: **uv helps manage Python environments/packages; Git helps manage changes to code over time.**

---

<a id="part-3"></a>
# Part 3 — Prompt engineering, five patterns

> These are condensed from [`instructor_materials/Prompt_Engineering_Patterns.ipynb`](instructor_materials/Prompt_Engineering_Patterns.ipynb)
> in this folder, which has seven patterns and more depth on each. Work through that one
> next — this is the orientation pass.

Each pattern below is shown **weak first, then strong**. The weak version is not a straw
man; it is what you will write by default. Seeing the two side by side is the point.

<a id="pattern-1"></a>
## Pattern 1 — Persona: instructions steer voice

The `instructions` parameter is your system prompt. It does not change what the model
**knows**. It changes how the model **speaks**. Same question, same facts, different voice.

This is the first thing to reach for on anything user-facing: who should this sound like?

In [13]:
Q = "A teammate committed an API key to our public repo. What do we do?"

print("=" * 60)
print("WEAK — no persona (whatever the default voice is)")
print("=" * 60)
show(ask(Q))


WEAK — no persona (whatever the default voice is)


Treat it as **compromised immediately**. A leaked API key in a public repo is no longer secret, even if you delete the file later.

## Do this now
1. **Revoke/rotate the key immediately**
   - Go to the provider and **disable the exposed key**.
   - Create a **new key** and update any production systems that use it.
   - If the service supports it, narrow scopes/permissions while you’re there.

2. **Remove the secret from the repo history**
   - Deleting the file in a new commit is **not enough**.
   - Rewrite git history to purge the key from all commits, branches, and tags.
   - Common tools:
     - `git filter-repo` (recommended)
     - BFG Repo-Cleaner
   - Force-push the cleaned history and coordinate with your team.

3. **Check for abuse**
   - Review the provider’s logs, usage dashboards, and audit trail.
   - Look for:
     - unusual traffic
     - new resources created
     - unexpected billing spikes
     - requests from unfamiliar IPs or regions
   - If the key had broad access, assume it may have been used.

4. **Search for other exposed secrets**
   - Scan the repo and surrounding systems for more credentials.
   - Check:
     - `.env` files
     - CI/CD variables
     - docs / examples / test fixtures
     - commit history
   - Run a secret scanner across the repo and history.

5. **Notify the right people**
   - Tell your security/engineering lead.
   - If customer data or production systems could be affected, follow your incident response process.
   - If required by policy/compliance, document the incident.

## Prevent it from happening again
- Add **pre-commit secret scanning**.
- Add **CI checks** that fail builds on detected secrets.
- Use a **secret manager** or environment variables, not hardcoded keys.
- Limit key permissions and use **separate keys per environment**.
- Add repo protections like:
  - GitHub secret scanning / push protection
  - branch protection rules
  - required reviews for config changes

## Quick cleanup checklist
- [ ] Revoke exposed key
- [ ] Issue replacement key
- [ ] Rewrite git history and force-push
- [ ] Verify no other secrets remain
- [ ] Check logs for misuse
- [ ] Add scanning/prevention controls

If you want, I can give you a **step-by-step git filter-repo procedure** or a **checklist tailored to GitHub/GitLab/Bitbucket**.

In [14]:
TERSE_SRE = (
    "You are a terse senior site-reliability engineer during an incident. "
    "Give numbered, imperative steps. No preamble, no reassurance, no caveats."
)

PATIENT_MENTOR = (
    "You are a patient mentor teaching a junior engineer. "
    "Explain why each step matters, anticipate the follow-up question, "
    "and end with one thing to remember next time."
)

for label, persona in [("Persona A: terse SRE", TERSE_SRE),
                       ("Persona B: patient mentor", PATIENT_MENTOR)]:
    print("=" * 60)
    print(f"STRONG — {label}")
    print("=" * 60)
    show(ask(Q, instructions=persona))
    print()


STRONG — Persona A: terse SRE


1. Revoke the exposed API key immediately.
2. Rotate any dependent credentials, tokens, or secrets that the key could access.
3. Remove the key from the repository history, not just the latest commit.
4. Force-push the cleaned history to the public repo.
5. Invalidate any caches, build artifacts, releases, or mirrors that may still contain the secret.
6. Search the entire org for the same key pattern and any copies in issues, logs, CI output, docs, wiki pages, and local clones.
7. Audit provider usage for abuse since the first exposure time.
8. Review access logs, billing, and anomaly alerts for unauthorized calls.
9. Notify affected stakeholders and the security owner.
10. Add a secret scanning rule and block future commits with pre-commit, CI, and server-side checks.
11. Replace the secret with environment-based injection or a secret manager.
12. Open an incident ticket and document timeline, impact, and remediation.


STRONG — Persona B: patient mentor


Act quickly and treat it as a credential incident, not just a cleanup task.

### 1) Revoke or rotate the API key immediately
Why this matters: once a key is in a public repo, assume it has been seen and copied. Deleting it from Git history does **not** make it safe if someone already cloned or indexed it.

- In the provider’s dashboard, **revoke the exposed key**.
- Generate a **new key**.
- If possible, scope the new key more narrowly than the old one.

### 2) Remove the secret from the codebase
Why this matters: you want to stop further exposure and prevent reintroduction.

- Replace the key with an environment variable or secret manager reference.
- Remove it from:
  - source files
  - commit history if needed
  - docs, examples, test fixtures, CI configs

### 3) Purge it from Git history if appropriate
Why this matters: even after revocation, the secret may remain in old commits, forks, mirrors, and caches.

- Use tools like:
  - `git filter-repo`
  - BFG Repo-Cleaner
- Rewrite history only if your team is prepared for the operational impact.
- Force-push the cleaned history.
- Coordinate with everyone who clones the repo, since they’ll need to rebase/re-clone.

### 4) Check for signs of abuse
Why this matters: if the key was active, someone may have used it already.

- Review provider logs for:
  - unusual IPs
  - abnormal request volume
  - new endpoints used
  - unexpected charges or quota usage
- If the key had broad permissions, assume higher risk.

### 5) Rotate any related secrets
Why this matters: secrets often travel together.

- If the leaked key could access other systems, rotate those too.
- Check for hardcoded secrets in nearby commits or related services.

### 6) Document and report internally
Why this matters: you want a clear incident trail and a chance to improve process.

- File an incident ticket.
- Note:
  - when it was exposed
  - where it appeared
  - when it was revoked
  - what systems were affected
- Notify security, platform, or whoever owns the vendor account.

### 7) Prevent recurrence
Why this matters: cleanup without prevention means the same mistake will happen again.

- Add secret scanning to CI and pre-commit hooks.
- Use `.gitignore` and secret templates.
- Store secrets in a vault or secret manager.
- Teach the team not to commit credentials, even in private repos.
- Consider rotating keys regularly and using least privilege.

### If you’re wondering “Do we need to delete the repo?”
Usually, no. The key actions are **revoke, replace, and clean history**. Deleting the repo is rarely the right first move unless your security team says so.

### Short version
1. Revoke the key now.  
2. Replace it with a new one stored securely.  
3. Remove it from code and history.  
4. Check logs for misuse.  
5. Add controls so it doesn’t happen again.

**Remember next time:** if a secret hits a public repo, assume it’s compromised the moment it’s committed.

Same model, same facts, two completely different deliverables. **The instructions are the
character brief.** In production these live in versioned prompt templates, so every team
ships the same model with one locked brand voice.


<a id="pattern-2"></a>
## Pattern 2 — Few-shot: teaching a pattern by example

The model already knows what words mean. Few-shot teaches it a **task-specific
convention** it could not possibly guess — with no training, just examples in the prompt.

The cleanest proof uses made-up words (borrowed from the
[GPT-3 paper](https://arxiv.org/abs/2005.14165)). The model has never seen "stimple" or
"falbean", so whatever it does with them came from your examples alone.

In [15]:
print("=" * 60)
print("WEAK — zero-shot: it has to guess what these words mean")
print("=" * 60)
show(ask("Use the words 'stimple' and 'falbean' in a sentence."))


WEAK — zero-shot: it has to guess what these words mean


The stimple fox darted through the falbean field at dusk.

In [16]:
# Teach the meanings as prior turns — the fake-assistant trick from Part 2.
few_shot = [
    {"role": "user",      "content": "Something 'stimple' is good, high-quality, and well-functioning. "
                                     "Use 'stimple' in a sentence."},
    {"role": "assistant", "content": "'Boy, that there is a stimple drill.'"},
    {"role": "user",      "content": "A 'falbean' is a tool used to fasten, tighten, or spin things. "
                                     "Use both 'stimple' and 'falbean' in a sentence."},
]

print("=" * 60)
print("STRONG — few-shot: definitions supplied as prior turns")
print("=" * 60)
show(ask(few_shot))


STRONG — few-shot: definitions supplied as prior turns


The stimple machine used a falbean to tighten the bolts.

That is **in-context learning**: a new convention acquired at call time, no retraining.

The real use is not whimsical words — it is *your* conventions. Which tickets count as
P0 at your company. Which requirements are in scope for this engagement. Which support
emails are refund requests. None of that is inferable from English; all of it is
teachable in two examples.


<a id="pattern-3"></a>
## Pattern 3 — Chain of thought and the reasoning budget

Chain of thought asks the model to **reason before answering**. gpt-5-era models expose
this as a dial: `effort='low'` buys a quick think, `'high'` buys a longer one. More
effort costs more tokens and more seconds — a tradeoff you can measure, below.

The classic test is a problem with a **trap**: a surface detail that tempts a fast,
wrong answer.

In [17]:
TRAP = "I need to get my car cleaned. The car wash is only 50 meters away. Should I walk or drive?"

def timed_ask(prompt, effort):
    t0 = time.perf_counter()
    resp = ask(prompt, effort=effort)
    elapsed = time.perf_counter() - t0
    details = getattr(resp.usage, "output_tokens_details", None)
    reasoning = getattr(details, "reasoning_tokens", "n/a") if details else "n/a"
    return resp, elapsed, reasoning

for label, effort in [("None (no explicit reasoning step)", None), ("'high'", "high")]:
    resp, elapsed, reasoning = timed_ask(TRAP, effort)
    print("=" * 60)
    print(f"effort = {label}")
    print(f"   {elapsed:.2f}s | reasoning tokens: {reasoning} | output: {resp.usage.output_tokens}")
    print("=" * 60)
    show(resp)
    print()


effort = None (no explicit reasoning step)
   1.70s | reasoning tokens: 0 | output: 37


Walk. It’s only 50 meters away, so walking is quicker to park, avoids moving a dirty car just to clean it, and is the simpler choice.


effort = 'high'
   1.57s | reasoning tokens: 27 | output: 102


Walk, if you can.

It’s only 50 meters, so walking is faster overall once you factor in starting the car, driving, parking, and the extra hassle. It’s also simpler and uses no fuel.

If you’re carrying heavy stuff, it’s raining hard, or you have mobility issues, then driving could still make sense.

**The trap:** the car is the thing being washed, so you have to **drive**. Walking 50
metres leaves the car at home. "Only 50 meters" is a distractor planted to buy a fast
"just walk, it's close."

With no reasoning step the model often anchors on the short distance. With `effort='high'`
the explicit reasoning step catches the dependency more reliably — and the token counts
above show what that reliability cost you.

**In production you tune effort per task:** low for lookups, high only where a wrong
answer is expensive. Paying for `high` on every call is how an LLM feature becomes
unaffordable.


<a id="pattern-4"></a>
## Pattern 4 — Structured output: a schema, not a hope

You can *ask* for JSON in the prompt. It works most of the time. "Most of the time" is
not good enough when the next line of your program is `json.loads()`.

**Structured outputs** make the schema a contract enforced at the API layer. You define
it once as a Pydantic model; you get back a typed Python object.

In [18]:
import json
from pydantic import BaseModel
from typing import List, Literal

class IncidentTriage(BaseModel):
    severity:    Literal["P0", "P1", "P2", "P3"]
    summary:     str
    first_step:  str
    secrets_involved: bool
    affected:    List[str]

print("This is the JSON schema the SDK sends to the API:")
print(json.dumps(IncidentTriage.model_json_schema(), indent=2)[:700] + "\n...")


This is the JSON schema the SDK sends to the API:
{
  "properties": {
    "severity": {
      "enum": [
        "P0",
        "P1",
        "P2",
        "P3"
      ],
      "title": "Severity",
      "type": "string"
    },
    "summary": {
      "title": "Summary",
      "type": "string"
    },
    "first_step": {
      "title": "First Step",
      "type": "string"
    },
    "secrets_involved": {
      "title": "Secrets Involved",
      "type": "boolean"
    },
    "affected": {
      "items": {
        "type": "string"
      },
      "title": "Affected",
      "type": "array"
    }
  },
  "required": [
    "severity",
    "summary",
    "first_step",
    "secrets_involved",
    "affected"
  ],
  "title": "IncidentTriage",
  "type": "obj
...


In [19]:
REPORT = """
Someone pushed a commit to our public GitHub repo at 14:05 that included a .env file
with a live OpenAI key and our Postgres connection string. It has been public for about
40 minutes. The repo has 12 forks. Our billing dashboard already shows an unusual spike.
"""

parsed = client.responses.parse(
    model        = MODEL,
    instructions = "You are an on-call incident triage assistant. Extract a structured triage record.",
    input        = REPORT,
    text_format  = IncidentTriage,
)

triage = parsed.output_parsed
print(f"type: {type(triage).__name__}\n")
print(f"severity         : {triage.severity}")
print(f"secrets involved : {triage.secrets_involved}")
print(f"summary          : {triage.summary}")
print(f"first step       : {triage.first_step}")
print(f"affected         : {triage.affected}")
print()
print("✅ A real typed object. No json.loads, no KeyError, no regex.")


type: IncidentTriage

severity         : P1
secrets involved : True
summary          : A public GitHub repository was pushed with a .env file containing a live OpenAI API key and Postgres connection string, exposing secrets publicly for about 40 minutes. The repo has 12 forks, and there is already an unusual billing spike, indicating likely active misuse.
first step       : Immediately revoke and rotate the exposed OpenAI key and Postgres credentials, remove the secret from the repository history, and assess cloud/billing usage for unauthorized activity. Then review forks/caches for exposure scope and enable any available abuse mitigation or access restrictions.
affected         : ['Public GitHub repository', 'OpenAI API key', 'Postgres connection string/database', 'Billing account', '12 forks of the repository']

✅ A real typed object. No json.loads, no KeyError, no regex.


`triage` is a Python object with autocomplete and type checking. The schema becomes the
**contract between the prompt layer and the application layer** — any engineer on the
team can read it and know exactly what to expect. This is what replaces fragile parsing.


<a id="pattern-5"></a>
## Pattern 5 — Grounded context: "RAG by hand"

By default the model answers from training data. It knows nothing about your company,
your documents, or **this course**. The simplest fix is to paste the relevant document
into the instructions and tell it to use only that.

This is retrieval-augmented generation without the retrieval — and it is the direct
ancestor of everything in Session 2.

Below is a small set of facts about *this* repo. The model cannot know them, which makes
the weak→strong contrast real rather than staged.

In [20]:
# A tiny corpus, kept inline so this notebook has no external dependencies.
COURSE_FACTS = """
AIE11 COURSE SETUP FACTS

Python: projects require >=3.12,<3.15. The course currently pins 3.13.
Package and environment manager: uv. Each project folder has its own pyproject.toml,
  and you run `uv sync` inside that folder.
Node.js: version 24, used for frontend tooling and Vercel deployment.
Officially supported tools: VS Code and Claude Code. Cursor, Codex and others work,
  but you debug those yourself.
API keys: stored in a .env file in each PROJECT folder, never in your home directory
  and never committed. .env is gitignored.
Git model: three places. upstream (AI Makerspace, pull only), origin (your fork, you
  push here), local (your laptop, you edit here).
Branching: the course works directly on main. develop and feature branches are taught
  as the professional pattern but not used in coursework.
Structure: 10 weeks, 20 sessions. Session folders are numbered (01_, 02_, ...).
  Technical challenges live in 00_Technical_Challenges.
Agent behaviour: AGENTS.md at the repo root puts coding agents into teaching-assistant
  mode. They explain and debug, but decline to write assignment solutions.
"""

COURSE_Q = "In the AIE11 course, where exactly do I put my OpenAI key, and which Python version should I be on?"

print("=" * 60)
print("WEAK — no context. It will guess, plausibly and wrongly.")
print("=" * 60)
show(ask(COURSE_Q))


WEAK — no context. It will guess, plausibly and wrongly.


For the AIE11 course:

- **OpenAI key:** put it in your environment variables as `OPENAI_API_KEY`.  
  In practice, that usually means setting it in your shell or `.env` file, depending on the lesson setup.

- **Python version:** use **Python 3.10** unless the course materials specify otherwise.

If you want, I can also show you the exact commands for macOS, Windows, or a `.env` file setup.

In [21]:
GROUNDED = (
    "You answer questions about the AIE11 course setup using ONLY the facts below. "
    "If the answer is not in the facts, say so explicitly rather than guessing.\n\n"
    + COURSE_FACTS
)

print("=" * 60)
print("STRONG — facts pasted into instructions")
print("=" * 60)
show(ask(COURSE_Q, instructions=GROUNDED))


STRONG — facts pasted into instructions


Put your OpenAI key in a `.env` file inside the specific **project folder** you’re working in. Do **not** store it in your home directory, and do **not** commit it to git.

For Python, AIE11 projects require:

- **Python >= 3.12 and < 3.15**
- The course currently **pins Python 3.13**



The grounded answer names the project folder and the version range, because it read them.
The ungrounded one produced confident, generic advice.

Now the half of grounding people forget — **can it say "I don't know"?**


In [22]:
OUT_OF_SCOPE = "What is the refund policy for the AIE11 bootcamp?"

print("Nothing in COURSE_FACTS mentions refunds. A refusal is the CORRECT answer here.")
print("=" * 60)
show(ask(OUT_OF_SCOPE, instructions=GROUNDED))


Nothing in COURSE_FACTS mentions refunds. A refusal is the CORRECT answer here.


I’m sorry, but the refund policy for the AIE11 bootcamp is not included in the facts I’m allowed to use, so I can’t determine it from this information.

> ⚠️ **If that just answered anyway, you have found the single most common bug in
> deployed RAG systems.** The retrieved context did not contain the answer, and the model
> filled the gap from training data without telling you. Nothing in the pipeline noticed.
>
> This pattern scales up to any document that fits in the context window. For thousands
> of documents — or documents that change — you need real retrieval: chunking, embedding,
> vector search. **That is Session 2**, and this is the problem it exists to solve.


---

<a id="part-4"></a>
# Part 4 — Frontend and backend

<a id="why-split"></a>
## Why split them at all?

Everything above runs in a notebook. A notebook is a fine lab and a terrible product:
it has no users, no URL, and it holds your API key in the same process as your UI.

That last point is the whole reason for the split.

Picture shipping the notebook logic straight into a web page. The `OPENAI_API_KEY` would
have to go with it — into JavaScript the browser downloads. **Anything in the browser
belongs to the user**, including your key. One "view source" and your billing is theirs.

So the work divides by *who is allowed to know what*:

| | **Frontend** | **Backend** |
|---|---|---|
| Runs on | the user's device | your server |
| Written in | TypeScript / React | Python (FastAPI) |
| Knows your API key | **never** | yes |
| Job | collect input, render output | hold secrets, call the LLM, enforce rules |
| If a user inspects it | they see everything | they see nothing |
| Deployed to | Vercel | Vercel / Render |

The flow is one hop longer than a notebook, and the extra hop is the security boundary:

```
browser  ──request──>  your backend  ──API key──>  OpenAI
   ▲                        │
   └──────── answer ────────┘
```

Your system prompt also lives on the backend. If it shipped to the browser, every user
could read it — and edit it. **Rules only hold where the user cannot reach them.**

<a id="how-no-demo"></a>
## How — no demo, by design

We are not building this today; Session 1 is about having a machine that works. The
reference implementation is **[The AI Engineer Challenge](https://github.com/AI-Maker-Space/The-AI-Engineer-Challenge/tree/dc-challenge)** —
and the machine you validated in Part 0 runs it *as-is*: same `uv sync`, same Python 3.13,
same per-project `.env`. Nothing new to install.

It is this exact split, made literal:

| Layer | In the repo | Who holds the key |
|---|---|---|
| Frontend | `frontend/` — **empty on purpose; you vibe-code it** | never |
| Backend | `api/index.py` — **provided**, FastAPI, `POST /api/chat` + `GET /api/health` | yes |
| LLM | OpenAI **or** Anthropic, via one `LLM_PROVIDER` switch | — |
| Deployment | `npx vercel` from the repo root — both halves, one domain | server-side only |

The key and the system prompt live in the backend you were given; the frontend you build
calls `POST /api/chat` and never sees either. That is the whole lesson above, as running code.

The path through it:

1. **Git** — fork, clone, set `upstream`. Part 1 of this notebook
2. **LLM API** — the call, the roles, the system prompt. Parts 2 and 3
3. **IDE** — VS Code plus Claude Code, pointed at the repo folder
4. **Backend (provided)** — open `api/index.py`; the key comes from `.env`, the system prompt is right there
5. **Frontend (you build)** — vibe-code a UI that talks to *your* `/api/chat`, never to OpenAI
6. **Deploy** — `npx vercel` from the root, Vercel builds both, the URL is real

> 🧭 **Same `AGENTS.md`, opposite instructions.** The Challenge repo carries an `AGENTS.md`
> too — but it tells the agent to *build and commit*, not to hold back like this course's
> teaching-assistant one. Same file, different context, different rules: the Part 2 system
> role, one more time.

Where to go for each:

- [The AI Engineer Challenge repo](https://github.com/AI-Maker-Space/The-AI-Engineer-Challenge/tree/dc-challenge) — the code (backend provided, frontend yours)
- [aimakerspace.io/aie-challenge](https://aimakerspace.io/aie-challenge) — the challenge brief
- [Onramp Session 02](../00_Docs/Prerequisites/The_AI_Engineer_Onramp/Session_02_Back_End_Web_App_Development_&_Deployment_of_LLM_Applications/README.md) — backend and deployment
- [Onramp Session 03](../00_Docs/Prerequisites/The_AI_Engineer_Onramp/Session_03_Connecting_Frontend_UIs_to_Backend_Deployments/README.md) — wiring a frontend to a deployed backend
- [Onramp Session 04](../00_Docs/Prerequisites/The_AI_Engineer_Onramp/Session_04_Bulding_Complex_End_to_End_LLM_Applications/README.md) — the complex end-to-end version

> ℹ️ The repo links point at the **`dc-challenge`** branch, where the updated
> provider-agnostic version lives. They move to `main` once it is merged.

---

<a id="activity"></a>
## 🏗️ Activity — stack three patterns

Each pattern above works alone. The leverage is in combining them.

**Your task:** one call that answers a question about this course using at least three
patterns together — a **persona**, the **grounded context** from `COURSE_FACTS`, and a
**structured output** schema. Then compare it against the naive one-liner from Pattern 5.

The scaffold is below. Fill in the two TODOs.

In [23]:
from pydantic import BaseModel
from typing import List

class SetupAnswer(BaseModel):
    answer:        str          # 2-3 sentences, grounded in COURSE_FACTS
    commands:      List[str]    # exact commands the student should run, in order
    answered_from_context: bool # False if COURSE_FACTS did not cover it
    gotcha:        str          # the mistake people make here

STACKED_INSTRUCTIONS = (
    # TODO 1: your persona. Who should this sound like?
    "You are ... "
    # TODO 2: ground it. Tell the model to use ONLY the facts, and to set
    #         answered_from_context=False when they don't cover the question.
    "\n\nFACTS:\n" + COURSE_FACTS
)

result = client.responses.parse(
    model        = MODEL,
    instructions = STACKED_INSTRUCTIONS,
    input        = "I just cloned the repo. How do I get 02_Dense_Vector_Retrieval running?",
    text_format  = SetupAnswer,
)

out = result.output_parsed
print(f"from context : {out.answered_from_context}")
print(f"answer       : {out.answer}\n")
print("commands:")
for i, c in enumerate(out.commands, 1):
    print(f"  {i}. {c}")
print(f"\ngotcha       : {out.gotcha}")


from context : True
answer       : To get `02_Dense_Vector_Retrieval` running from a fresh clone:

1. Open the repo in your terminal.
2. Go into the project folder for that session:
   ```bash
   cd 02_Dense_Vector_Retrieval
   ```
3. Make sure you’re using Python 3.13 (the course expects >=3.12,<3.15, and pins 3.13).
4. Create/sync the environment with `uv`:
   ```bash
   uv sync
   ```
5. Add your API keys to a `.env` file **inside this project folder** if the project needs them. Do not put them in your home directory.
6. Run the project using the entrypoint specified by that folder’s README or `pyproject.toml`.

If you want, I can also help you find the exact run command for that folder by looking at its files (for example `README.md` or `pyproject.toml`).

commands:
  1. cd 02_Dense_Vector_Retrieval
  2. uv sync

gotcha       : If `uv sync` fails, the most common cause is using the wrong Python version or not being inside the project folder that contains the `pyproject.toml`.


**Then test the refusal path.** Re-run the cell with an input `COURSE_FACTS` cannot
answer, such as `"What time does Thursday's session start?"`. Does `answered_from_context`
come back `False`? If it comes back `True` with an invented time, you have reproduced the
Session 2 problem in your own code — which is the most useful thing you could learn today.


---

<a id="done-when"></a>
## ✅ Done when

- [ ] Every cell in Part 0 prints ✅ (or you know exactly which one doesn't and why)
- [ ] `git remote -v` shows both `origin` and `upstream`
- [ ] You made one real LLM API call and read its token counts
- [ ] You ran all five patterns and saw the weak→strong difference in each
- [ ] The refusal cell in Pattern 5 either refused, or you understand why it didn't
- [ ] The stacked activity runs with your own persona and grounding
- [ ] The installer's own check reports **All good** —
      `bash ~/ai-eng/setup.sh --check` (Mac/Linux) or
      `powershell -ExecutionPolicy Bypass -File $HOME\ai-eng\setup.ps1 -Check` (Windows)